## **OceanOSSE:** Development Notebook

### **Description:**

Notebook to develop & evaluate the core data pipeline in OceanOSSE.

### **Created By:**

Ollie Tooth (oliver.tooth@noc.ac.uk)

In [ ]:
import logging

import xarray as xr

from OceanOSSE.gridding.regridder import MockRegridder
from OceanOSSE.io.dataloader import NetCDFDataLoader
from OceanOSSE.io.datawriter import NetCDFDataWriter
from OceanOSSE.sampling.sampler import MockObsSampler

In [ ]:
logger = logging.getLogger(__name__)

logging.basicConfig(
    format="⦿══⦿  OceanOSSE  ⦿══⦿  ║ %(levelname)10s ║ %(asctime)s ║ %(message)s",
    level=logging.INFO,
    datefmt="%Y-%m-%d %H:%M:%S",
    handlers=[logging.FileHandler("OceanOSSE.log"), logging.StreamHandler()],
)

### Example configuration .toml file read as dict

In [ ]:
config = {
    "domain": {
        "dimensions": {"lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "nav_lev",
        },
        "variables": {
            "tmask": {
                "path": "/dssgfs01/scratch/npd/simulations/Domains/eORCA025/eORCA025_ERA5v1_domain_cfg_mesh_mask_util.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "inputs": {
        "dimensions": {"time": "time_counter", "lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "deptht",
            "time": "time_counter",
        },
        "variables": {
            "thetao_con": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
            "so_abs": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "climatology": {
        "read_climatology": False,
        "dimensions": {"month": "time_counter", "lev": "deptht", "j": "y", "i": "x"},
        "coordinates": {
            "lat": "nav_lat",
            "lon": "nav_lon",
            "depth": "deptht",
            "month": "time_counter",
        },
        "variables": {
            "thetao_con": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
            "so_abs": {
                "path": "/dssgfs01/scratch/npd/simulations/eORCA1_ERA5_v1/eORCA1_ERA5_1m_grid_T_202*.nc",
                "open_kwargs": {"engine": "netcdf4"},
            },
        },
    },
    "sampling": {
        "name": "test",
        "error_kernels": [{"name": "test", "kwargs": {"argument": None}}],
    },
    "regridding": {"name": "test", "kwargs": {"argument": None}},
    "outputs": {
        "output_dir": "/dssgfs01/working/otooth/Software/OceanOSSE/OceanOSSE",
        "output_name": "OceanOSSE_TEST",
        "date_format": "M",
        "chunks": {"time_counter": 12},
        "writer_kwargs": {"unlimited_dims": "time_counter", "mode": "w"},
    },
}

config

### Data Loaders - Input Scalar Fields

In [ ]:
dataloader = NetCDFDataLoader.from_config(config=config, table="inputs")

In [ ]:
# View coordinate mapping:
dataloader._coordinates

In [ ]:
ds = dataloader.load_data()
ds = ds.sel(time=slice("2023-01", "2023-12"))
ds

### Data Loaders - Climatologies

In [ ]:
ds_clim = dataloader.compute_monthly_climatology().sel(month=ds['time'].dt.month)
ds_clim

In [ ]:
clim_writer = NetCDFDataWriter.from_config(config=config, climatology=True)
clim_output_filepath = clim_writer.write_data(ds=ds_clim)
clim_output_filepath

### Sampling - MockObsSampler

In [ ]:
sampler = MockObsSampler.from_config(config=config)
ds_obs = sampler.sample(ds=ds)
ds_obs

In [ ]:
# Must return the input Dataset unmodified:
ds_obs.identical(ds)

### Regridding - MockRegridder

In [ ]:
regridder = MockRegridder.from_config(config=config)
ds_obs_regridded = regridder.regrid(ds=ds_obs)
ds_obs_regridded

In [ ]:
# Must return the input Dataset unmodified:
ds_obs_regridded.identical(ds_obs)

### Data Writers - NetCDFWriter

In [ ]:
datawriter = NetCDFDataWriter.from_config(config=config)
datawriter.write_data(ds=ds_obs_regridded)
datawriter

In [ ]:
# Verify output Dataset:
ds_out = xr.open_dataset(
    "/dssgfs01/working/otooth/Software/OceanOSSE/OceanOSSE//OceanOSSE_TEST_2023-12-2023-12.nc"
)

ds_out